In [1]:
import sys
from pathlib import Path

root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [2]:
import torch
import numpy as np
from gensim.models import KeyedVectors
from src.utils import load_config, load_vocab, save_embedding_matrix

In [3]:
# Nạp cấu hình và từ vựng Flickr30k
config, config_dir = load_config()

vocab_path = config_dir / config["paths"]["vocab"]
vocab = load_vocab(vocab_path)
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 7736


In [4]:
# Nạp mô hình GoogleNews Word2Vec (~3.64 GB)
word2vec_path = config_dir / config["paths"]["word2vec"]
print(f"Loading Word2Vec model from: {word2vec_path}")

kv = KeyedVectors.load_word2vec_format(str(word2vec_path), binary=True)
print(f"Successfully loaded {len(kv):,} words with dimension {kv.vector_size}.")

Loading Word2Vec model from: D:\Documents\Projects\image-captioning\data\raw\GoogleNews-vectors-negative300.bin
Successfully loaded 3,000,000 words with dimension 300.


In [5]:
# Trích xuất ma trận nhúng cho vocabulary của Flickr30k
embed_size = kv.vector_size
vocab_size = len(vocab)
embedding_matrix = np.zeros((vocab_size, embed_size), dtype=np.float32)

# Tính mean và std từ mẫu vector trong Word2Vec để khởi tạo các từ OOV cân bằng
sample_keys = list(kv.key_to_index.keys())[:10000]
sample_vectors = np.array([kv[w] for w in sample_keys])
emb_mean = float(sample_vectors.mean())
emb_std = float(sample_vectors.std())
print(f"Word2Vec statistics: mean = {emb_mean:.4f}, std = {emb_std:.4f}")

np.random.seed(config["train"]["seed"])  

found_words = 0
oov_words = []

for idx in range(vocab_size):
    word = vocab.idx2word[idx]

    # 1. <pad> vector là vector 0
    if word == "<pad>":
        embedding_matrix[idx] = np.zeros(embed_size, dtype=np.float32)
        continue

    # 2. Token đặc biệt <start>, <end>, <unk>
    if word in ("<start>", "<end>", "<unk>"):
        embedding_matrix[idx] = np.random.normal(emb_mean, emb_std, size=(embed_size,))
        continue

    # 3. Tìm từ trong Word2Vec (thử dạng gốc, lowercase, capitalize, uppercase)
    matched = None
    for candidate in [word, word.lower(), word.capitalize(), word.upper()]:
        if candidate in kv:
            matched = kv[candidate]
            break

    if matched is not None:
        embedding_matrix[idx] = matched
        found_words += 1
    else:
        embedding_matrix[idx] = np.random.normal(emb_mean, emb_std, size=(embed_size,))
        oov_words.append(word)

total_content_words = vocab_size - 4  # Trừ 4 token đặc biệt
coverage = (found_words / total_content_words) * 100

print(f"\nTotal content words (excluding 4 special tokens): {total_content_words}")
print(f"Words found in Word2Vec: {found_words} ({coverage:.2f}%)")
print(f"OOV words: {len(oov_words)} ({100 - coverage:.2f}%)")
print(f"Some sample OOV words: {oov_words[:25]}")

Word2Vec statistics: mean = -0.0034, std = 0.1771

Total content words (excluding 4 special tokens): 7732
Words found in Word2Vec: 7546 (97.59%)
OOV words: 186 (2.41%)
Some sample OOV words: ['.', ',', 'dark-haired', 'brown-haired', 'tri-colored', '(', ')', 't-shirt', '``', 'short-sleeved', 'dark-skinned', '?', 'gray-haired', 'long-sleeved', 'long-sleeve', 'middle-aged', ';', '13', 'blond-hair', 'long-haired', 'red-hair', 'red-haired', 'college-aged', '-', 'blond-haired']


In [6]:
# Chuyển sang PyTorch Tensor và lưu xuống data/processed/embedding_matrix.pt
embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)
print(f"Embedding tensor shape: {embedding_tensor.shape}")
print(f"Embedding tensor dtype: {embedding_tensor.dtype}")

save_embedding_matrix(embedding_tensor)
emb_path = config_dir / config["paths"]["embedding_matrix"]
print(f"Successfully saved in {emb_path}")

Embedding tensor shape: torch.Size([7736, 300])
Embedding tensor dtype: torch.float32
Successfully saved in D:\Documents\Projects\image-captioning\data\processed\embedding_matrix.pt


In [7]:
# Sanity check: kiểm tra nạp lại
from src.utils import load_embedding_matrix

loaded = load_embedding_matrix(emb_path)
print(f"Loaded embedding matrix shape: {loaded.shape}")
assert loaded.shape == (len(vocab), 300), f"Shape mismatch! Expected ({len(vocab)}, 300), got {loaded.shape}"
assert torch.all(loaded[0] == 0), "Vector of <pad> should be all zeros!"
print("All checks passed!")

Loaded embedding matrix shape: torch.Size([7736, 300])
All checks passed!
